# 🚀 Russian IT Community LLM & RAG Quickstart
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wwewtech/russian-it-community-corpus/blob/main/notebooks/01_quickstart_inference_rag_lora.ipynb)

This interactive notebook demonstrates how to:
1. Load the **Russian IT Community Corpus** (`wwewtech/russian-it-community-corpus`) from Hugging Face.
2. Load pre-trained **LoRA Adapters** (`wwewtech/russian-it-community-lora`).
3. Execute **RAG semantic retrieval** and generate high-precision technical answers.

In [ ]:
# 1. Install Required Dependencies
!pip install -q datasets transformers peft accelerate bitsandbytes pyarrow

In [ ]:
# 2. Load Dataset from Hugging Face Hub
from datasets import load_dataset

print("Loading SFT Dialogues dataset from Hugging Face...")
sft_ds = load_dataset("wwewtech/russian-it-community-corpus", "sft_dialogues", split="train")
print(f"Loaded {len(sft_ds)} dialogues!")
print("Sample dialogue:", sft_ds[0])

In [ ]:
# 3. Load Foundation Model & LoRA Adapter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("Model loaded successfully!")

In [ ]:
# 4. Generate Expert Technical Response
prompt = "Как устроена репликация WAL в PostgreSQL и как избежать лагов репликации при high-load?"
messages = [
    {"role": "system", "content": "Ты — старший ведущий инженер и архитектор баз данных."},
    {"role": "user", "content": prompt}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))